# Laboratorio: K-Means

**Pregunta:** ¿qué estructura encontramos entre observaciones sin usar etiquetas como respuesta? En Iris cada fila es una flor y las cuatro medidas (cm) describen sépalos y pétalos. El mismo proceso se extiende luego a erupciones de Old Faithful y a clientes mayoristas.

**Ruta:** problema → datos → exploración → variables → escala → modelo → experimentación → selección → evaluación → perfiles → interpretación.


## Objetivos de aprendizaje

- Explicar centroides y distancias cuadráticas.
- Preparar variables y justificar decisiones sin recurrir a la clase conocida.
- Comparar configuraciones, perfilar grupos en unidades originales y reconocer límites de la evaluación interna.


## 1. Problema y método

En aprendizaje no supervisado no conocemos el grupo correcto de cada fila. Las etiquetas botánicas de Iris se reservan únicamente como contraste descriptivo al final: no se utilizan para elegir variables, entrenar ni seleccionar el número de grupos.

K-Means agrupa minimizando la suma de distancias cuadráticas a los centroides. Su inercia siempre disminuye al aumentar K; silhouette combina cohesión y separación.


## 2. Configuración y datos

Importamos solo las herramientas que utiliza este método. Fijamos una semilla para los algoritmos con inicialización aleatoria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')
from sklearn.cluster import KMeans


### Comprensión de Iris

Revisamos tamaño, tipos, faltantes, duplicados y rango de las mediciones. Una fila es una flor; no usamos `target` en la matriz de entrenamiento.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.data.copy()
y_reference = iris.target.copy()
display(df.head())
print("Filas y columnas:", df.shape)
print("Faltantes:", df.isna().sum().to_dict())
print("Duplicados:", df.duplicated().sum())
display(df.describe().round(2))


**Lectura:** las variables miden centímetros, pero tienen rangos y dispersión distintos. Comprobar faltantes y duplicados antes del ajuste evita que decisiones implícitas alteren las distancias. Un duplicado puede ser una medición genuina, por lo que aquí no se elimina automáticamente.

## 3. Exploración y elección de variables

Visualizamos longitud y ancho de pétalo como una proyección interpretable de dos dimensiones. El entrenamiento utilizará las cuatro mediciones; la gráfica no determina la partición.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df['petal length (cm)'], df['petal width (cm)'], alpha=.7)
ax.set(xlabel='Longitud de pétalo (cm)', ylabel='Ancho de pétalo (cm)', title='Iris: proyección exploratoria')
plt.show()


**Lectura:** se observa una zona de pétalos pequeños y otra con mayor solapamiento. Esta vista parcial no demuestra que existan exactamente dos o tres grupos en el espacio de cuatro variables.

### Variables para el modelo

Conservamos las cuatro medidas continuas porque describen dimensiones morfológicas relevantes; excluimos el nombre de especie y cualquier identificador. La correlación puede hacer que dos medidas relacionadas tengan más peso conjunto; analizaremos esa limitación.

In [ ]:
features = list(df.columns)
X = df[features].copy()
display(X.corr().round(2))


## 4. Representación para el modelo

Como el método depende de distancias o covarianzas, estandarizamos cada medida. Conservamos `X` original para interpretar posteriormente los grupos en centímetros.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=X.index)
display(X_scaled_df.describe().loc[['mean', 'std']].round(2))


## 5. Modelo inicial

Probamos K=3 como punto de partida, sin asumir que coincide con las especies. La función objetivo es $J=\sum_k\sum_{x_i\in C_k}\|x_i-\mu_k\|^2$, donde $\mu_k$ es el centroide. `fit_predict` ajusta centroides y asigna filas.

In [ ]:
initial_model = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE)
initial_labels = initial_model.fit_predict(X_scaled)
print('Inercia inicial:', round(initial_model.inertia_, 2))
display(pd.DataFrame(scaler.inverse_transform(initial_model.cluster_centers_), columns=features).round(2))


## 6. Experimentación: codo y silhouette

La inercia mide compactación (menor es mejor, pero decrece por construcción al aumentar K). Silhouette compara cohesión y separación (mayor es mejor); exige al menos dos grupos. Ninguna métrica revela por sí sola un K verdadero.
**Métricas complementarias:** Calinski–Harabasz compara separación y dispersión (mayor suele indicar mejor separación); Davies–Bouldin resume semejanza entre grupos (menor suele ser mejor). Ambas favorecen ciertas geometrías y deben leerse junto a tamaños, perfiles y el objetivo del análisis. En GMM no reemplazan AIC/BIC: solo describen la partición rígida derivada de las probabilidades.


In [ ]:
results = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    results.append({'k': k, 'inertia': model.inertia_,
                    'silhouette': silhouette_score(X_scaled, labels), 'calinski_harabasz': calinski_harabasz_score(X_scaled, labels), 'davies_bouldin': davies_bouldin_score(X_scaled, labels)})
results = pd.DataFrame(results)
display(results.round(3))


### Análisis gráfico de configuraciones — Iris

Comparamos las curvas de K/componentes para detectar mejoras que se estabilizan, máximos o mínimos. Las escalas de los criterios son distintas: cada métrica tiene su propio eje. AIC/BIC admiten K=1; los índices basados en separación empiezan en K=2. El gráfico orienta la selección; perfiles y estabilidad completan la interpretación.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = np.ravel(axes)
axes[0].plot(results['k'], results['inertia'], marker='o')
axes[0].set(xlabel='Número de clusters', ylabel='Inercia', title='Inercia (menor con K; buscar desaceleración)')
axes[1].plot(results['k'], results['silhouette'], marker='o')
axes[1].set(xlabel='Número de clusters', ylabel='Silhouette', title='Silhouette (mayor)')
axes[2].plot(results['k'], results['calinski_harabasz'], marker='o')
axes[2].set(xlabel='Número de clusters', ylabel='Calinski–Harabasz', title='Calinski–Harabasz (mayor)')
axes[3].plot(results['k'], results['davies_bouldin'], marker='o')
axes[3].set(xlabel='Número de clusters', ylabel='Davies–Bouldin', title='Davies–Bouldin (menor)')
plt.tight_layout(); plt.show()


**Lectura:** identifica si las métricas coinciden en una configuración o señalan compromisos diferentes. Un óptimo en una curva describe este espacio de variables y esta muestra; comprueba tamaños y perfiles antes de interpretar los grupos.

### Comparación visual

Buscamos si las mejoras de compactación se atenúan y si silhouette favorece una solución interpretable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(results.k, results.inertia, marker='o')
axes[0].set(xlabel='K', ylabel='Inercia', title='Método del codo')
axes[1].plot(results.k, results.silhouette, marker='o')
axes[1].set(xlabel='K', ylabel='Silhouette', title='Cohesión y separación')
plt.tight_layout(); plt.show()


## 7. Selección de configuración

Seleccionamos un candidato por el criterio declarado; contrastaremos tamaño, perfiles y plausibilidad antes de darle significado. Una métrica interna no valida que los grupos sean entidades naturales.

In [ ]:
selected_k = int(results.loc[results.silhouette.idxmax(), 'k'])
print('K candidato por silhouette:', selected_k)


## 8. Ajuste y evaluación final

Ajustamos el candidato usando la misma representación estandarizada. Los números de grupo son identificadores arbitrarios. Las métricas se calculan sobre las variables usadas en el ajuste.

In [ ]:
final_model = KMeans(n_clusters=selected_k, n_init=20, random_state=RANDOM_STATE)
labels = final_model.fit_predict(X_scaled)
print('Silhouette final:', round(silhouette_score(X_scaled, labels), 3))
centers = pd.DataFrame(scaler.inverse_transform(final_model.cluster_centers_), columns=features)
display(centers.round(2))


## 9. Proyección para visualizar

Mostramos dos medidas de pétalo originales, aunque el ajuste utilizó las cuatro variables. El solapamiento aquí no describe necesariamente la separación en cuatro dimensiones.

In [ ]:
plot_df = df.copy()
plot_df['cluster'] = labels
fig, ax = plt.subplots(figsize=(7, 5))
for cluster, part in plot_df.groupby('cluster'):
    ax.scatter(part['petal length (cm)'], part['petal width (cm)'], label=f'Cluster {cluster}', alpha=.75)
ax.set(xlabel='Longitud de pétalo (cm)', ylabel='Ancho de pétalo (cm)', title='Iris: grupos en dos variables originales')
ax.legend(); plt.tight_layout(); plt.show()


## 10. Perfiles e interpretación

Volvemos a las medidas originales para describir los grupos en centímetros. Comparamos tamaños y medias; las especies conocidas se cruzan al final como referencia externa, nunca como criterio de ajuste o selección.

In [ ]:
profile = plot_df.groupby('cluster')[features].agg(['mean', 'median']).round(2)
sizes = plot_df.groupby('cluster').size().rename('n')
display(sizes)
display(profile)
display(pd.crosstab(plot_df['cluster'], y_reference.rename('especie_codificada')))


### Síntesis basada en los resultados

Identificamos extremos de longitud de pétalo sin asumir que el número de cluster expresa un orden. La tabla de medias permite comprobar la afirmación en el resultado calculado.

In [ ]:
petal_means = plot_df.groupby('cluster')['petal length (cm)'].mean()
print(f'Cluster con menor longitud media de pétalo: {petal_means.idxmin()} ({petal_means.min():.2f} cm)')
print(f'Cluster con mayor longitud media de pétalo: {petal_means.idxmax()} ({petal_means.max():.2f} cm)')


## 11. Ejercicio 2 — Old Faithful: erupciones del géiser

**Pregunta:** ¿se distinguen comportamientos de erupción por duración (minutos) y tiempo de espera hasta la siguiente erupción (minutos)? Cada fila es un episodio; no existe etiqueta de cluster. Fuente: dataset `faithful` de R, publicado por Rdatasets. Este ejercicio repite el workflow completo con dos variables observables. Requiere acceso a la fuente CSV.


### Datos y exploración

Comprobamos registros, ausencias y escalas. Una dispersión en unidades originales permite reconocer estructura preliminar sin asignar grupos.

In [ ]:
faithful_url = 'https://vincentarelbundock.github.io/Rdatasets/csv/datasets/faithful.csv'
faithful = pd.read_csv(faithful_url).drop(columns='rownames')
print('Dimensiones:', faithful.shape, 'Nulos:', faithful.isna().sum().to_dict())
display(faithful.head())
display(faithful.describe().round(2))
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(faithful.eruptions, faithful.waiting, alpha=.7)
ax.set(xlabel='Duración (min)', ylabel='Espera (min)', title='Old Faithful: observaciones originales')
plt.show()


### Selección de variables y preparación

Ambas mediciones son relevantes; omitimos el índice del archivo. La distancia o covarianza se estima con variables estandarizadas. Los perfiles volverán a minutos originales.

In [ ]:
faithful_features = ['eruptions', 'waiting']
faithful_X = faithful[faithful_features].copy()
faithful_scaler = StandardScaler()
faithful_scaled = faithful_scaler.fit_transform(faithful_X)
display(pd.DataFrame(faithful_scaled, columns=faithful_features).describe().loc[['mean', 'std']].round(2))


### Experimento de K

Contrastamos inercia y silhouette; el codo no tiene fórmula única. Dejamos visible cada ajuste y su evaluación.
**Métricas complementarias:** Calinski–Harabasz compara separación y dispersión (mayor suele indicar mejor separación); Davies–Bouldin resume semejanza entre grupos (menor suele ser mejor). Ambas favorecen ciertas geometrías y deben leerse junto a tamaños, perfiles y el objetivo del análisis. En GMM no reemplazan AIC/BIC: solo describen la partición rígida derivada de las probabilidades.


In [ ]:
faithful_scores = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    candidate_labels = model.fit_predict(faithful_scaled)
    faithful_scores.append({'k': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(faithful_scaled, candidate_labels), 'calinski_harabasz': calinski_harabasz_score(faithful_scaled, candidate_labels), 'davies_bouldin': davies_bouldin_score(faithful_scaled, candidate_labels)})
faithful_scores = pd.DataFrame(faithful_scores)
display(faithful_scores.round(3))


### Análisis gráfico de configuraciones — Old Faithful

Comparamos las curvas de K/componentes para detectar mejoras que se estabilizan, máximos o mínimos. Las escalas de los criterios son distintas: cada métrica tiene su propio eje. AIC/BIC admiten K=1; los índices basados en separación empiezan en K=2. El gráfico orienta la selección; perfiles y estabilidad completan la interpretación.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = np.ravel(axes)
axes[0].plot(faithful_scores['k'], faithful_scores['inertia'], marker='o')
axes[0].set(xlabel='Número de clusters', ylabel='Inercia', title='Inercia (menor con K; buscar desaceleración)')
axes[1].plot(faithful_scores['k'], faithful_scores['silhouette'], marker='o')
axes[1].set(xlabel='Número de clusters', ylabel='Silhouette', title='Silhouette (mayor)')
axes[2].plot(faithful_scores['k'], faithful_scores['calinski_harabasz'], marker='o')
axes[2].set(xlabel='Número de clusters', ylabel='Calinski–Harabasz', title='Calinski–Harabasz (mayor)')
axes[3].plot(faithful_scores['k'], faithful_scores['davies_bouldin'], marker='o')
axes[3].set(xlabel='Número de clusters', ylabel='Davies–Bouldin', title='Davies–Bouldin (menor)')
plt.tight_layout(); plt.show()


**Lectura:** identifica si las métricas coinciden en una configuración o señalan compromisos diferentes. Un óptimo en una curva describe este espacio de variables y esta muestra; comprueba tamaños y perfiles antes de interpretar los grupos.

### Selección, ajuste y centroides

Usamos el máximo silhouette como candidato y contrastamos el perfil con la dispersión. Los centroides se expresan otra vez en minutos.

In [ ]:
faithful_k = int(faithful_scores.loc[faithful_scores.silhouette.idxmax(), 'k'])
faithful_model = KMeans(n_clusters=faithful_k, n_init=20, random_state=RANDOM_STATE)
faithful_labels = faithful_model.fit_predict(faithful_scaled)
print('K:', faithful_k, 'Silhouette:', round(silhouette_score(faithful_scaled, faithful_labels), 3))
display(pd.DataFrame(faithful_scaler.inverse_transform(faithful_model.cluster_centers_), columns=faithful_features).round(2))


### Visualización, perfil y lectura

El gráfico usa las dos variables modeladas; los promedios y tamaños originales permiten describir erupciones cortas o largas sin atribuir significado ordinal a los identificadores.

In [ ]:
faithful_profile = faithful.copy()
faithful_profile['cluster'] = faithful_labels
display(faithful_profile.groupby('cluster').agg({'eruptions':['size', 'mean'], 'waiting':'mean'}).round(2))
fig, ax = plt.subplots(figsize=(7, 5))
for cluster, part in faithful_profile.groupby('cluster'):
    ax.scatter(part.eruptions, part.waiting, label=f'Cluster {cluster}', alpha=.7)
ax.set(xlabel='Duración (min)', ylabel='Espera (min)', title=f'Old Faithful: K-Means (K={faithful_k})')
ax.legend(); plt.show()
print('Grupo de menor duración media:', faithful_profile.groupby('cluster').eruptions.mean().idxmin())


**Conclusión guiada:** identifica en la tabla qué grupo tiene la duración y la espera media mayores. Valora si hay solapamiento en el gráfico y si la solución responde a la pregunta; ningún algoritmo garantiza que los regímenes de erupción sean categorías naturales.

## 12. Ejercicio 3 — Wholesale Customers: segmentos de gasto

**Pregunta:** ¿existen patrones de gasto anual entre clientes mayoristas? Cada fila es un cliente y las seis categorías de gasto son las variables del modelo. `Channel` y `Region` describen segmentos después del ajuste; no son etiquetas de entrenamiento. Fuente: UCI Machine Learning Repository, dataset 292. Se usa su CSV público; requiere conexión de red.


### Datos y exploración

Comprobamos esquema, ausencias y asimetría antes de decidir transformaciones. Las magnitudes originales se conservan para perfilado.

In [ ]:
wholesale_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv'
wholesale = pd.read_csv(wholesale_url)
spend_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
print('Dimensiones:', wholesale.shape, 'Nulos:', wholesale.isna().sum().to_dict())
display(wholesale.head())
display(wholesale[spend_features].describe().round(1))


### Variables y transformación

Excluimos `Channel` y `Region` del ajuste. `log1p` reduce el peso de gastos extremos sin descartar clientes; después escalamos las seis variables. La distancia del modelo se calcula en este espacio transformado.

In [ ]:
wholesale_X = wholesale[spend_features].copy()
wholesale_log = np.log1p(wholesale_X)
wholesale_scaler = StandardScaler()
wholesale_scaled = wholesale_scaler.fit_transform(wholesale_log)
display(wholesale_log.describe().loc[['mean', 'std']].round(2))


### K alternativos

Evaluamos compactación e intensidad de separación, teniendo presente que pueden favorecer configuraciones diferentes.

In [ ]:
wholesale_scores = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    candidate_labels = model.fit_predict(wholesale_scaled)
    wholesale_scores.append({'k': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(wholesale_scaled, candidate_labels), 'calinski_harabasz': calinski_harabasz_score(wholesale_scaled, candidate_labels), 'davies_bouldin': davies_bouldin_score(wholesale_scaled, candidate_labels)})
wholesale_scores = pd.DataFrame(wholesale_scores)
display(wholesale_scores.round(3))


### Análisis gráfico de configuraciones — Wholesale Customers

Comparamos las curvas de K/componentes para detectar mejoras que se estabilizan, máximos o mínimos. Las escalas de los criterios son distintas: cada métrica tiene su propio eje. AIC/BIC admiten K=1; los índices basados en separación empiezan en K=2. El gráfico orienta la selección; perfiles y estabilidad completan la interpretación.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = np.ravel(axes)
axes[0].plot(wholesale_scores['k'], wholesale_scores['inertia'], marker='o')
axes[0].set(xlabel='Número de clusters', ylabel='Inercia', title='Inercia (menor con K; buscar desaceleración)')
axes[1].plot(wholesale_scores['k'], wholesale_scores['silhouette'], marker='o')
axes[1].set(xlabel='Número de clusters', ylabel='Silhouette', title='Silhouette (mayor)')
axes[2].plot(wholesale_scores['k'], wholesale_scores['calinski_harabasz'], marker='o')
axes[2].set(xlabel='Número de clusters', ylabel='Calinski–Harabasz', title='Calinski–Harabasz (mayor)')
axes[3].plot(wholesale_scores['k'], wholesale_scores['davies_bouldin'], marker='o')
axes[3].set(xlabel='Número de clusters', ylabel='Davies–Bouldin', title='Davies–Bouldin (menor)')
plt.tight_layout(); plt.show()


**Lectura:** identifica si las métricas coinciden en una configuración o señalan compromisos diferentes. Un óptimo en una curva describe este espacio de variables y esta muestra; comprueba tamaños y perfiles antes de interpretar los grupos.

### Configuración candidata y ajuste final

Aplicamos el criterio principal explicado arriba y ajustamos el modelo final. Este valor es una decisión para el laboratorio y puede revisarse si los perfiles resultan inestables o poco útiles.

In [ ]:
wholesale_k = int(wholesale_scores.loc[wholesale_scores.silhouette.idxmax(), 'k'])
wholesale_model = KMeans(n_clusters=wholesale_k, n_init=20, random_state=RANDOM_STATE)
wholesale_labels = wholesale_model.fit_predict(wholesale_scaled)
print('K candidato:', wholesale_k)


### Perfiles de negocio

Se agrupan los **gastos originales** y se comparan Channel y Region después del ajuste. Cada número de grupo es arbitrario; los cruces descriptivos no demuestran causas.

In [ ]:
wholesale_profile = wholesale.copy()
wholesale_profile['cluster'] = wholesale_labels
display(wholesale_profile.groupby('cluster')[spend_features].agg(['mean', 'median']).round(1))
display(wholesale_profile.groupby('cluster').size().rename('clientes'))
display(pd.crosstab(wholesale_profile.cluster, wholesale_profile.Channel, normalize='index').round(2))
display(pd.crosstab(wholesale_profile.cluster, wholesale_profile.Region, normalize='index').round(2))


### Distribuciones por cluster: boxplots de gasto

Cada panel compara la mediana, dispersión y valores extremos de una categoría de gasto entre clusters. Mostramos `log1p(gasto)` solo para visualizar distribuciones muy asimétricas; las tablas previas mantienen el gasto original. La escala horizontal identifica el cluster y la vertical la categoría transformada. Una diferencia de medias puede deberse a unos pocos clientes extremos: contrastamos siempre con mediana y caja.


In [ ]:
wholesale_plot = wholesale_profile[['cluster'] + spend_features].copy()
wholesale_plot[spend_features] = np.log1p(wholesale_plot[spend_features])
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flat, spend_features):
    sns.boxplot(data=wholesale_plot, x='cluster', y=feature, ax=ax, color='#77aadd', showfliers=True)
    ax.set(xlabel='Cluster', ylabel=f'log(1 + {feature})', title=f'Distribución por cluster: {feature}')
fig.suptitle('Wholesale Customers — variación del gasto dentro de cada cluster', y=1.01)
plt.tight_layout(); plt.show()


**Lectura de los boxplots:** compara las medianas y el ancho de las cajas para cada feature; observa si algún grupo solo se diferencia por clientes extremos. Las cajas superpuestas indican que esa variable por sí sola discrimina poco, aunque el modelo use las seis variables conjuntamente.

### Densidades KDE por cluster

Las curvas suavizadas permiten comparar forma, multimodalidad y solapamiento de cada feature. Se estiman sobre `log1p(gasto)` y se normalizan **dentro de cada cluster**, de modo que la altura no representa tamaño del grupo. La suavización puede sugerir modos artificiales en clusters pequeños: contrastamos con tamaños y boxplots.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flat, spend_features):
    for cluster, part in wholesale_plot.groupby('cluster'):
        values = part[feature].dropna()
        if len(values) >= 3 and values.nunique() >= 3:
            sns.kdeplot(x=values, ax=ax, label=f'Cluster {cluster}', common_norm=False, warn_singular=False)
    ax.set(xlabel=f'log(1 + {feature})', ylabel='Densidad estimada', title=f'Densidad de {feature}')
    if ax.lines:
        ax.legend(title='Grupo')
fig.suptitle('Wholesale Customers — forma y solapamiento de distribuciones', y=1.01)
plt.tight_layout(); plt.show()


**Interpretación conjunta:** usa las curvas para identificar features con densidades bien separadas y aquellas con fuerte superposición. Contrasta ese patrón con medianas, valores extremos y tamaños; una diferencia visible en una variable no implica que ella sola explique toda la asignación multivariable.


### Proyección parcial e interpretación

Mostramos Grocery y Detergents_Paper en escala log para inspección; el modelo utilizó las seis categorías. Leemos el segmento con mayor gasto medio en Grocery según el resultado real.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for cluster, part in wholesale_profile.groupby('cluster'):
    ax.scatter(np.log1p(part.Grocery), np.log1p(part.Detergents_Paper), label=f'Cluster {cluster}', alpha=.7)
ax.set(xlabel='log(1 + Grocery)', ylabel='log(1 + Detergents_Paper)', title=f'Wholesale: K-Means (K={wholesale_k})')
ax.legend(); plt.show()
print('Grupo de mayor Grocery promedio:', wholesale_profile.groupby('cluster').Grocery.mean().idxmax())


**Conclusión guiada:** compara proporciones de Channel y Region y el tamaño de cada grupo. Un segmento pequeño puede reflejar clientes con gasto atípico; revisa medianas además de medias antes de asignar una etiqueta comercial. El resultado depende de `log1p`, escalado, variables y configuración.

## 13. Limitaciones y conclusiones

La selección por silhouette es una propuesta, sujeta a inspeccionar codo y perfiles. K-Means es sensible a escala, valores extremos, inicialización y grupos no aproximadamente convexos. Los centroides representan medias y deben leerse en cm después de invertir la escala.

Los perfiles calculados arriba identifican qué grupos tienen pétalos cortos o largos y permiten discutir si la partición responde a la pregunta inicial. No debemos atribuir causalidad ni universalidad a estas agrupaciones.


## 14. Ejercicios sin solución

1. **Conceptual:** explica qué información se pierde al observar solo dos de las cuatro medidas.
2. **Código:** repite el ajuste usando exclusivamente las dos variables de pétalo y compara los perfiles.
3. **Experimento:** compara dos configuraciones y razona cuál elegirías con métricas y tamaños.
4. **Interpretación:** redacta un perfil por grupo en unidades originales y detecta solapamientos.
5. **Transferencia:** aplica el workflow a Old Faithful o Wholesale Customers; documenta la fuente, faltantes y efecto del escalado.


## 15. Referencias y datos

- Scikit-learn: documentación de Iris, preprocesamiento y del estimador usado.
- Iris: Fisher (1936); datos disponibles mediante `sklearn.datasets.load_iris`.
- Old Faithful: `datasets::faithful` (R); fuente original utilizada: Rdatasets.
- Wholesale Customers: UCI Machine Learning Repository, conjunto 292.
